In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!cp "/content/drive/MyDrive/Forecasting/master_long_hourly_test_2014_05_12.csv" "/content/master_long_hourly_test_2014_05_12.csv"
!cp "/content/drive/MyDrive/Forecasting/master_long_hourly_train_2012_2013.csv" "/content/master_long_hourly_train_2012_2013.csv"
!cp "/content/drive/MyDrive/Forecasting/master_long_hourly_validation_2014_01_04.csv" "/content/master_long_hourly_validation_2014_01_04.csv"
!cp "/content/drive/MyDrive/Forecasting/calendar_features_hourly.csv" "/content/calendar_features_hourly.csv"

In [3]:
from pathlib import Path

# Input files — produced by preprocessing + data-split teammate
TRAIN_PATH    = "/content/master_long_hourly_train_2012_2013.csv"
VAL_PATH      = "/content/master_long_hourly_validation_2014_01_04.csv"
TEST_PATH     = "/content/master_long_hourly_test_2014_05_12.csv"
CALENDAR_PATH = "/content/calendar_features_hourly.csv"

# Output — share this file with teammates for alignment
# SELECTED_CLIENTS_PATH = Path("selected_clients_sarimax.csv")
MODEL_DIR = Path("models")
MODEL_DIR.mkdir(exist_ok=True)

# Sampling
# RANDOM_SEED = 42
# N_CLIENTS   = 5    # increase carefully — SARIMAX is slow per client

# Model
SEASON_LENGTH = 24  # hourly data → daily seasonality

In [4]:
# IMPORTS
import warnings
warnings.filterwarnings('ignore')

import joblib
import numpy as np
import pandas as pd
from prophet import Prophet

In [5]:
# LOAD CLIENTS
# Reuse the same clients as SARIMAX so metrics are directly comparable
# selected = pd.read_csv(SELECTED_CLIENTS_PATH)['client_id'].tolist()
# print(f'Using {len(selected)} clients: {selected}')

# Use all clients from the training data instead of sampling
df_tmp = pd.read_csv(TRAIN_PATH)
selected = df_tmp['client_id'].unique().tolist()
print(f'Using all {len(selected)} clients from training data')

Using all 156 clients from training data


In [6]:
# PREPARE DATA
# Prophet has built-in daily, weekly, and yearly seasonality via Fourier series,
# so cyclic encodings (hour_sin/cos, dow_sin/cos, etc.) would be redundant.
# is_weekend is kept as an extra regressor - it captures a structural break
# (flat vs. peaked consumption) that weekly seasonality alone may smooth over.
EXOG_COLS = ['is_weekend']

calendar = pd.read_csv(CALENDAR_PATH, parse_dates=['timestamp'])


def load_split(path, clients, calendar, exog_cols):
    df = pd.read_csv(path, parse_dates=['timestamp'])
    df = df[df['client_id'].isin(clients)].copy()
    df = df.merge(calendar[['timestamp'] + exog_cols], on='timestamp', how='left')
    df = df.rename(columns={'client_id': 'unique_id', 'timestamp': 'ds'})
    return df.sort_values(['unique_id', 'ds']).reset_index(drop=True)


train = load_split(TRAIN_PATH, selected, calendar, EXOG_COLS)
val   = load_split(VAL_PATH,   selected, calendar, EXOG_COLS)
test  = load_split(TEST_PATH,  selected, calendar, EXOG_COLS)

print(f"train : {train.shape}  {train['ds'].min()} -> {train['ds'].max()}")
print(f"val   : {val.shape}    {val['ds'].min()} -> {val['ds'].max()}")
print(f"test  : {test.shape}   {test['ds'].min()} -> {test['ds'].max()}")

train : (2721888, 4)  2012-01-01 00:00:00 -> 2013-12-31 23:00:00
val   : (445536, 4)    2014-01-01 00:00:00 -> 2014-04-30 23:00:00
test  : (913536, 4)   2014-05-01 00:00:00 -> 2014-12-31 23:00:00


In [7]:
# FORECAST (Train & Validation)
def predict_split(models, split_df, exog_cols):
    preds = []
    # Grouping by unique_id is faster than filtering for each client
    for uid, grp in split_df.groupby('unique_id'):
        if uid not in models:
            continue
        m = models[uid]
        future = grp[['ds'] + exog_cols].copy()
        forecast = m.predict(future)[['ds', 'yhat']].rename(columns={'yhat': 'Prophet'})
        forecast['unique_id'] = uid
        preds.append(forecast)
    return pd.concat(preds, ignore_index=True)

# EVALUATION METRICS
def compute_metrics(df, target_col='y', pred_col='Prophet'):
    y    = df[target_col].values
    yhat = df[pred_col].values
    mse  = np.mean((y - yhat) ** 2)
    mae  = np.mean(np.abs(y - yhat))
    wape = np.sum(np.abs(y - yhat)) / np.sum(np.abs(y))
    return {'MSE': round(mse, 4), 'MAE': round(mae, 4), 'WAPE': round(wape, 4)}


def per_client_metrics(df, target_col='y', pred_col='Prophet'):
    rows = []
    for uid, grp in df.groupby('unique_id'):
        rows.append({'client_id': uid, **compute_metrics(grp, target_col, pred_col)})
    rows.append({'client_id': 'OVERALL', **compute_metrics(df, target_col, pred_col)})
    return pd.DataFrame(rows)

In [13]:
# TRAIN PROPHET
# One Prophet model is fit per client - Prophet is not a panel model.
# Seasonality mode: additive works well unless consumption variance scales with level.
def fit_prophet(df, exog_cols):
    models = {}
    for uid, grp in df.groupby('unique_id'):
        m = Prophet(
            daily_seasonality=True,
            weekly_seasonality=True,
            yearly_seasonality=True,
            seasonality_mode='additive',
        )
        for col in exog_cols:
            m.add_regressor(col)
        m.fit(grp[['ds', 'y'] + exog_cols])
        models[uid] = m
        print(f'  Fitted: {uid}')
    return models

In [ ]:
print('Training Prophet models...')
prophet_models = fit_prophet(train, EXOG_COLS)

joblib.dump(prophet_models, MODEL_DIR / 'prophet_val.joblib')
print('Training complete. Models saved → models/prophet_val.joblib')

Training Prophet models...
  Fitted: MT_124
  Fitted: MT_132
  Fitted: MT_156
  Fitted: MT_158
  Fitted: MT_159
  Fitted: MT_161
  Fitted: MT_162
  Fitted: MT_163
  Fitted: MT_166
  Fitted: MT_168
  Fitted: MT_169
  Fitted: MT_171
  Fitted: MT_172
  Fitted: MT_174
  Fitted: MT_175
  Fitted: MT_176
  Fitted: MT_180
  Fitted: MT_182
  Fitted: MT_183
  Fitted: MT_187
  Fitted: MT_188
  Fitted: MT_189
  Fitted: MT_190
  Fitted: MT_191
  Fitted: MT_192
  Fitted: MT_193
  Fitted: MT_194
  Fitted: MT_195
  Fitted: MT_196
  Fitted: MT_197
  Fitted: MT_198
  Fitted: MT_199
  Fitted: MT_200
  Fitted: MT_201
  Fitted: MT_202
  Fitted: MT_203
  Fitted: MT_204
  Fitted: MT_205
  Fitted: MT_206
  Fitted: MT_207
  Fitted: MT_208
  Fitted: MT_209
  Fitted: MT_210
  Fitted: MT_211
  Fitted: MT_212
  Fitted: MT_213
  Fitted: MT_214
  Fitted: MT_215
  Fitted: MT_216
  Fitted: MT_217
  Fitted: MT_218
  Fitted: MT_219
  Fitted: MT_220
  Fitted: MT_221
  Fitted: MT_222
  Fitted: MT_225
  Fitted: MT_226
  Fi

In [9]:
# LOAD SAVED MODELS
# Use this to load pre-trained models instead of re-running the training cell.

model_path = MODEL_DIR / 'prophet_val.joblib'

if model_path.exists():
    print(f"Loading models from {model_path}...")
    prophet_models = joblib.load(model_path)
    print(f"Successfully loaded {len(prophet_models)} models.")
else:
    print(f"File not found at {model_path}. Please run the training cell to generate the models.")

Loading models from models/prophet_val.joblib...
Successfully loaded 156 models.


In [10]:
print('Predicting on Training set...')
train_preds = predict_split(prophet_models, train, EXOG_COLS)
train_eval  = train[['unique_id', 'ds', 'y']].merge(train_preds, on=['unique_id', 'ds'])

print('Predicting on Validation set...')
val_preds = predict_split(prophet_models, val, EXOG_COLS)
val_eval  = val[['unique_id', 'ds', 'y']].merge(val_preds, on=['unique_id', 'ds'])

print(val_eval.head())

Predicting on Training set...
Predicting on Validation set...
  unique_id                  ds          y     Prophet
0    MT_124 2014-01-01 00:00:00  95.693780  305.275095
1    MT_124 2014-01-01 01:00:00  83.732056  280.589067
2    MT_124 2014-01-01 02:00:00  82.535890  238.627889
3    MT_124 2014-01-01 03:00:00  82.535890  181.716850
4    MT_124 2014-01-01 04:00:00  63.397133  124.410928


In [11]:
train_metrics = per_client_metrics(train_eval)
print('-- Training metrics --')
print(train_metrics.to_string(index=False))
print('\n')

val_metrics = per_client_metrics(val_eval)
print('-- Validation metrics --')
print(val_metrics.to_string(index=False))

-- Training metrics --
client_id          MSE       MAE   WAPE
   MT_124    1947.1266   33.3640 0.1217
   MT_132     125.7129    8.1900 0.5826
   MT_156     368.5904   15.7709 0.2160
   MT_158     946.6936   25.2073 0.3703
   MT_159     682.1916   21.2088 0.3985
   MT_161   38557.1585  137.1248 0.0792
   MT_162    5632.1216   59.7747 0.2145
   MT_163   41434.2622  152.2945 0.0661
   MT_166   12881.7801   82.3673 0.0651
   MT_168     246.0469   11.9979 0.0886
   MT_169      11.7678    2.5972 0.0664
   MT_171     427.0985   15.6642 0.0640
   MT_172     243.2254   11.8956 0.0986
   MT_174     223.7827   10.4107 0.0705
   MT_175     417.3802   15.8553 0.0974
   MT_176      79.5250    6.9112 0.0821
   MT_180     453.0849   16.4938 0.1052
   MT_182      19.3055    3.3661 0.0645
   MT_183      83.6817    6.8841 0.0807
   MT_187     163.1158   10.7184 0.2129
   MT_188      12.6690    2.9032 0.0854
   MT_189    2493.9327   38.0431 0.0924
   MT_190    1842.7993   31.9855 0.0820
   MT_191   24970

In [14]:
# TEST EVALUATION
# Retrain on full train+val, then predict on the held-out test set
train_val = pd.concat([train, val], ignore_index=True).sort_values(['unique_id', 'ds'])

print('Retraining on train+val...')
prophet_models_final = fit_prophet(train_val, EXOG_COLS)

joblib.dump(prophet_models_final, MODEL_DIR / 'prophet_final.joblib')
print('Final models saved → models/prophet_final.joblib')

test_preds = predict_split(prophet_models_final, test, EXOG_COLS)
test_eval  = test[['unique_id', 'ds', 'y']].merge(test_preds, on=['unique_id', 'ds'])

test_metrics = per_client_metrics(test_eval)
print('-- Test metrics --')
print(test_metrics.to_string(index=False))

Retraining on train+val...
  Fitted: MT_124
  Fitted: MT_132
  Fitted: MT_156
  Fitted: MT_158
  Fitted: MT_159
  Fitted: MT_161
  Fitted: MT_162
  Fitted: MT_163
  Fitted: MT_166
  Fitted: MT_168
  Fitted: MT_169
  Fitted: MT_171
  Fitted: MT_172
  Fitted: MT_174
  Fitted: MT_175
  Fitted: MT_176
  Fitted: MT_180
  Fitted: MT_182
  Fitted: MT_183
  Fitted: MT_187
  Fitted: MT_188
  Fitted: MT_189
  Fitted: MT_190
  Fitted: MT_191
  Fitted: MT_192
  Fitted: MT_193
  Fitted: MT_194
  Fitted: MT_195
  Fitted: MT_196
  Fitted: MT_197
  Fitted: MT_198
  Fitted: MT_199
  Fitted: MT_200
  Fitted: MT_201
  Fitted: MT_202
  Fitted: MT_203
  Fitted: MT_204
  Fitted: MT_205
  Fitted: MT_206
  Fitted: MT_207
  Fitted: MT_208
  Fitted: MT_209
  Fitted: MT_210
  Fitted: MT_211
  Fitted: MT_212
  Fitted: MT_213
  Fitted: MT_214
  Fitted: MT_215
  Fitted: MT_216
  Fitted: MT_217
  Fitted: MT_218
  Fitted: MT_219
  Fitted: MT_220
  Fitted: MT_221
  Fitted: MT_222
  Fitted: MT_225
  Fitted: MT_226
  Fi